In [1]:
import sys; sys.path.append('..')
import glob, os
from src import settings, TextRetrievalEngine, get_gemini_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain


c:\Users\kyle0\miniconda3\envs\rag_system\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# 하이브리드 기법을 제외한 순수 텍스트 리트리버 구축
pdf_files = glob.glob(os.path.join(settings.RAW_DATA_DIR, "*.pdf"))
engine = TextRetrievalEngine()
retriever, _ = engine.build_or_load(pdf_files, k=3)
print("✅ Baseline DB 준비 완료")


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:20: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  self.embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28440.32it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


⚡ 하드디스크에 저장된 DB와 문서를 불러옵니다!


c:\Users\kyle0\Develops\Financial_RAG_System\notebooks\..\src\retrieval\text_engine.py:35: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=db_dir, embedding_function=self.embedding_model)


✅ Baseline DB 준비 완료


In [3]:
query = "삼성전자의 2024년 연결재무제표 기준 당기순이익은 얼마인가요?"
llm = get_gemini_model()
prompt = ChatPromptTemplate.from_template("다음 문서를 바탕으로 답하세요:\n{context}\n질문: {input}")

chain = create_retrieval_chain(retriever, create_stuff_documents_chain(llm, prompt))
response = chain.invoke({"input": query})
print(f"🤖 답변: {response['answer']}")



🤖 답변: 제공된 문서에는 삼성전자의 2025년 연결기준 매출 및 영업이익에 대한 정보는 포함되어 있으나, **2024년 연결재무제표 기준 당기순이익에 대한 직접적인 수치는 명시되어 있지 않습니다.**

제공된 [삼성전자] 사업보고서(251페이지)의 '26. 주당이익' 항목에는 당기(2025년)의 당기순이익(보통주 29,607,505백만 원, 우선주 4,079,096백만 원)에 대한 정보만 기재되어 있습니다.
